# 💳 Credit Card Fraud Detection

## Problem Statement
Credit card fraud is a major financial threat, costing billions of dollars globally each year. The goal of this project is to build a machine learning model that can accurately detect fraudulent transactions from a dataset of real-world credit card activity. Early and accurate detection helps financial institutions minimize losses and protect customers.

## Dataset Overview
The dataset contains credit card transactions with features such as transaction amount, time, merchant category, cardholder demographics (age, gender, location), and a binary label `is_fraud` indicating whether the transaction was fraudulent.

---

In [1]:
# ============================================================
# CELL 1 — IMPORTING LIBRARIES
# ============================================================
# We import all the libraries we need upfront.
# pandas & numpy → data manipulation
# matplotlib & seaborn → visualizations
# datetime → to dynamically get the current year (fixes the hardcoded 2020 bug)
# warnings → to suppress non-critical output and keep the notebook clean

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("Libraries imported successfully")

Libraries imported successfully


---
## 1. Loading & Understanding the Data

In [2]:
# ============================================================
# CELL 2 — LOADING DATASET
# ============================================================
# Load the CSV file into a pandas DataFrame.
# UPDATE the path below to match where your CSV file is saved on your computer.

df = pd.read_csv(r"C:\Users\A S U S\aidi-1204-project\My project\credit_card_transactions.csv\credit_card_transactions.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\A S U S\\aidi-1204-project\\My project\\credit_card_transactions.csv\\credit_card_transactions.csv'

In [ ]:
# ============================================================
# CELL 3 — INITIAL DATA INSPECTION
# ============================================================
# Quick look at the first few rows to understand what the data looks like.
# shape → confirms how big the dataset is
# dtypes → tells us which columns are numeric vs text
# This is the first thing any interviewer expects you to do.

print("Shape:", df.shape)
print("\nColumn Types:")
print(df.dtypes)
df.head()

In [ ]:
# ============================================================
# CELL 4 — CHECK FOR MISSING VALUES
# ============================================================
# Checking how many null values exist per column.
# Any column with too many nulls needs to be dropped or imputed.

missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found")

In [ ]:
# ============================================================
# CELL 5 — DROP UNNECESSARY COLUMNS & RENAME
# ============================================================
# 'Unnamed: 0' is just a leftover index column from the CSV — safe to drop.
# 'merch_zipcode' has too many missing values and isn't useful for our model.
# We also rename columns to be more readable and consistent.

df.drop(['Unnamed: 0', 'merch_zipcode'], axis=1, inplace=True, errors='ignore')

df.rename(columns={
    'trans_date_trans_time': 'trans_date_time',
    'cc_num': 'credit_card_number',
    'amt': 'amount'
}, inplace=True)

print("Columns cleaned. Remaining columns:")
print(df.columns.tolist())

---
## 2. Exploratory Data Analysis (EDA)

EDA helps us understand patterns in the data *before* building any model. We want to understand: where does fraud happen, how much money is involved, and which features stand out.

In [ ]:
# ============================================================
# CELL 6 — CLASS IMBALANCE: FRAUD vs LEGITIMATE
# ============================================================
# This is one of the most important things to show in a fraud project.
# We visualize how many transactions are fraudulent vs legitimate.
# The chart shows the raw imbalance — this motivates our later resampling step.

fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print("Class Distribution:")
print(f"  Legitimate (0): {fraud_counts[0]:,}  ({fraud_pct[0]:.2f}%)")
print(f"  Fraud      (1): {fraud_counts[1]:,}  ({fraud_pct[1]:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values, color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Transaction Count by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Fraud vs Legitimate (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

**Observation:** The dataset is highly imbalanced — fraudulent transactions make up only a small percentage of all transactions. This is typical in real-world fraud data. If we trained a model as-is, it would learn to predict "legitimate" for everything and still get high accuracy. This is why we need to handle class imbalance before training.

In [ ]:
# ============================================================
# CELL 7 — TRANSACTION AMOUNT: FRAUD vs LEGITIMATE
# ============================================================
# We compare the transaction amounts between fraud and legitimate cases.
# The boxplot shows that fraud transactions often cluster at different amount ranges.
# The histogram helps visualize the distribution more clearly.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(x='is_fraud', y='amount', data=df, palette=['steelblue', 'tomato'], ax=axes[0])
axes[0].set_xticklabels(['Legitimate', 'Fraud'])
axes[0].set_title('Transaction Amount by Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Transaction Type')
axes[0].set_ylabel('Amount ($)')

# Histogram (capped at $1000 for readability)
df[df['is_fraud'] == 0]['amount'].clip(upper=1000).hist(bins=50, alpha=0.6, color='steelblue', label='Legitimate', ax=axes[1])
df[df['is_fraud'] == 1]['amount'].clip(upper=1000).hist(bins=50, alpha=0.6, color='tomato', label='Fraud', ax=axes[1])
axes[1].set_title('Amount Distribution (capped at $1000)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

**Observation:** Fraudulent transactions tend to have a different amount distribution compared to legitimate ones. This confirms that `amount` will be a useful feature for our model.

In [ ]:
# ============================================================
# CELL 8 — FRAUD BY MERCHANT CATEGORY
# ============================================================
# Here we look at which merchant categories have the highest fraud rates.
# We calculate fraud *rate* per category (not raw count) so high-volume
# categories don't automatically look worse than others.

category_fraud = df.groupby('category')['is_fraud'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
bars = plt.barh(category_fraud.index, category_fraud.values * 100,
                color=sns.color_palette('RdYlGn_r', len(category_fraud)))
plt.xlabel('Fraud Rate (%)')
plt.title('Fraud Rate by Merchant Category', fontsize=13, fontweight='bold')
for bar, val in zip(bars, category_fraud.values):
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val*100:.1f}%', va='center')
plt.tight_layout()
plt.show()

**Observation:** Some merchant categories have noticeably higher fraud rates. This suggests that `category` is an important predictor and is worth keeping in our model.

In [ ]:
# ============================================================
# CELL 9 — FRAUD BY TRANSACTION HOUR
# ============================================================
# We extract the hour of day from the transaction timestamp and check
# if fraud tends to happen at certain times (e.g., late at night).
# This gives us the `trans_hour` feature we'll use later in the model.

df['trans_date_time'] = pd.to_datetime(df['trans_date_time'])
df['trans_hour'] = df['trans_date_time'].dt.hour

hour_fraud = df.groupby('trans_hour')['is_fraud'].mean()

plt.figure(figsize=(12, 5))
plt.plot(hour_fraud.index, hour_fraud.values * 100, marker='o', color='tomato', linewidth=2)
plt.fill_between(hour_fraud.index, hour_fraud.values * 100, alpha=0.2, color='tomato')
plt.xticks(range(0, 24))
plt.xlabel('Hour of Day (0 = Midnight)')
plt.ylabel('Fraud Rate (%)')
plt.title('Fraud Rate by Hour of Day', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:** Fraud is significantly more common during late-night / early-morning hours (roughly 10 PM – 4 AM). This pattern motivates the `is_night` feature we will engineer in the next section.

In [ ]:
# ============================================================
# CELL 10 — FRAUD BY AGE GROUP
# ============================================================
# We temporarily calculate age here just for the EDA visualization.
# (The permanent age feature will be engineered properly in the next section.)
# Grouping by age bins shows whether certain age groups are targeted more.

df['dob'] = pd.to_datetime(df['dob'])
df['age_temp'] = datetime.now().year - df['dob'].dt.year
df['age_group'] = pd.cut(df['age_temp'], bins=[0, 25, 35, 45, 55, 65, 100],
                          labels=['<25', '25-35', '35-45', '45-55', '55-65', '65+'])

age_fraud = df.groupby('age_group')['is_fraud'].mean()

plt.figure(figsize=(10, 5))
bars = plt.bar(age_fraud.index.astype(str), age_fraud.values * 100,
               color=sns.color_palette('Blues_d', len(age_fraud)), edgecolor='black')
for bar, val in zip(bars, age_fraud.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val*100:.1f}%', ha='center', fontsize=10)
plt.xlabel('Age Group')
plt.ylabel('Fraud Rate (%)')
plt.title('Fraud Rate by Age Group', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

df.drop(columns=['age_temp', 'age_group'], inplace=True)  # clean up temporary columns

**Observation:** Fraud rates vary across age groups. Older cardholders may be targeted more frequently, which makes `age` a meaningful feature for our model.

---
## 3. Feature Engineering

We create new features that help the model better understand the data. Good feature engineering is often the difference between an average model and a great one.

In [ ]:
# ============================================================
# CELL 11 — FEATURE: AGE FROM DATE OF BIRTH
# ============================================================
# FIX: The original code used hardcoded 2020. We now use datetime.now().year
# so the age is always calculated relative to the current year.
# After extracting age, we drop the raw dob column since it's no longer needed.

df['age'] = datetime.now().year - df['dob'].dt.year
df = df.drop(columns=['dob'])

print("Age feature created. Sample values:")
print(df['age'].describe())

In [ ]:
# ============================================================
# CELL 12 — FEATURE: DISTANCE BETWEEN CARDHOLDER AND MERCHANT
# ============================================================
# We calculate the Euclidean distance between the cardholder's location
# (lat/long) and the merchant's location (merch_lat/merch_long).
# A large distance can be a signal of fraud — the card is being used
# far away from where the cardholder is located.

df['distance'] = np.sqrt(
    (df['lat'] - df['merch_lat'])**2 +
    (df['long'] - df['merch_long'])**2
)

print("Distance feature created. Sample:")
print(df['distance'].describe())

In [ ]:
# ============================================================
# CELL 13 — FEATURE: IS_NIGHT FLAG
# ============================================================
# Based on our EDA, fraud happens more often between 10 PM and 4 AM.
# We create a binary flag: 1 if the transaction was during those hours, 0 otherwise.
# Binary flags like this are simple but very effective for tree-based models.

df['is_night'] = df['trans_hour'].apply(lambda h: 1 if (h >= 22 or h <= 4) else 0)

print("is_night feature created.")
print(df['is_night'].value_counts())
print(f"\nFraud rate at night: {df[df['is_night']==1]['is_fraud'].mean()*100:.2f}%")
print(f"Fraud rate during day: {df[df['is_night']==0]['is_fraud'].mean()*100:.2f}%")

In [ ]:
# ============================================================
# CELL 14 — FEATURE: LOG TRANSFORM CITY POPULATION 
# ============================================================
# city_pop (city population) is likely heavily skewed — a few very large cities
# vs many small ones. Log transformation compresses this scale so the model
# isn't dominated by extreme values. We keep the original column for reference.

df['city_pop_log'] = np.log1p(df['city_pop'])  # log1p = log(1 + x), safe for 0 values

print("city_pop_log feature created.")
print(df[['city_pop', 'city_pop_log']].describe())

In [ ]:
# ============================================================
# CELL 15 — DROP COLUMNS NOT USEFUL FOR MODELING
# ============================================================
# These columns are either identifiers (like card number, name, address)
# or redundant after feature engineering (like raw lat/long after distance is made).
# Keeping them would confuse the model or cause privacy issues in production.

cols_to_drop = [
    'trans_date_time',
    'credit_card_number',
    'merchant',
    'first',
    'last',
    'street',
    'city',
    'zip',
    'job',
    'trans_num',
    'unix_time',
    'city_pop',   # replaced by city_pop_log
    'lat',        # replaced by distance feature
    'long',
    'merch_lat',
    'merch_long'
]

df = df.drop(columns=cols_to_drop, errors='ignore')
print("Columns after dropping identifiers:")
print(df.columns.tolist())

In [ ]:
# ============================================================
# CELL 16 — ENCODING CATEGORICAL COLUMNS
# ============================================================
# Machine learning models require numeric input — they can't process text.
# gender: simple binary map (M=1, F=0)
# category & state: one-hot encoding (get_dummies) creates a binary column
# for each unique value. drop_first=True avoids the "dummy variable trap".

df['gender'] = df['gender'].map({'M': 1, 'F': 0})
df = pd.get_dummies(df, columns=['category'], drop_first=True)
df = pd.get_dummies(df, columns=['state'], drop_first=True)

print("Encoding complete. Final shape:", df.shape)

---
## 4. Handling Class Imbalance

> ⚠️ **Important fix from original code:** We now split the data into train/test sets *first*, then upsample *only on the training set*. Upsampling before the split causes **data leakage** — synthetic copies of minority samples end up in both train and test, making the model look better than it really is.

In [ ]:
# ============================================================
# CELL 17 — TRAIN/TEST SPLIT FIRST  ← FIXED ORDER
# ============================================================
# We split the data BEFORE any resampling. This ensures the test set
# reflects real-world class distribution and hasn't been contaminated
# by upsampled data. random_state=42 ensures reproducibility.

from sklearn.model_selection import train_test_split
from sklearn.utils import resample

X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)  # stratify=y → keeps the same fraud ratio in both train and test

print(f"Train size: {X_train.shape[0]:,}")
print(f"Test size:  {X_test.shape[0]:,}")
print(f"\nFraud % in train: {y_train.mean()*100:.2f}%")
print(f"Fraud % in test:  {y_test.mean()*100:.2f}%")

In [ ]:
# ============================================================
# CELL 18 — UPSAMPLE ONLY THE TRAINING DATA  ← FIXED
# ============================================================
# Now we balance the training set by oversampling the minority (fraud) class.
# We use resample() to create synthetic copies of fraud cases until it matches
# the number of legitimate transactions. This is done ONLY on training data.
# The test set stays untouched so our evaluation is realistic.

train_df = pd.concat([X_train, y_train], axis=1)

majority = train_df[train_df['is_fraud'] == 0]
minority = train_df[train_df['is_fraud'] == 1]

minority_upsampled = resample(minority,
                               replace=True,
                               n_samples=len(majority),
                               random_state=42)

train_balanced = pd.concat([majority, minority_upsampled]).sample(frac=1, random_state=42)

X_train = train_balanced.drop('is_fraud', axis=1)
y_train = train_balanced['is_fraud']

print(f"After balancing — Train size: {X_train.shape[0]:,}")
print(f"Fraud cases in training: {y_train.sum():,}")
print(f"Legit cases in training: {(y_train==0).sum():,}")

---
## 5. Model Training

In [ ]:
# ============================================================
# CELL 19 — TRAIN RANDOM FOREST
# ============================================================
# Random Forest is an ensemble of decision trees — it votes across 100 trees
# to make a prediction. It handles non-linear patterns well and gives us
# feature importances. max_depth=10 prevents the trees from overfitting.

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)
print("Random Forest trained")

In [ ]:
# ============================================================
# CELL 20 — TRAIN LOGISTIC REGRESSION
# ============================================================
# Logistic Regression is a simpler, interpretable linear model.
# It works well as a baseline to compare against more complex models.
# class_weight='balanced' helps it handle the remaining imbalance.

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=500,
    solver='liblinear',
    class_weight='balanced',
    random_state=42
)
lr_model.fit(X_train, y_train)
print("Logistic Regression trained")

In [ ]:
# ============================================================
# CELL 21 — TRAIN XGBOOST 
# ============================================================
# XGBoost is a powerful gradient boosting algorithm — often top-performing
# on tabular data. scale_pos_weight compensates for class imbalance by
# telling XGBoost how much more to weight the minority (fraud) class.
# Adding XGBoost shows you know multiple model families, not just sklearn.

from xgboost import XGBClassifier

scale = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)
print("XGBoost trained")

---
## 6. Model Evaluation

For fraud detection, **accuracy alone is misleading** — a model that labels everything as "legit" can still score 99% accuracy on an imbalanced dataset. We focus on:
- **Precision**: of all transactions flagged as fraud, how many were actually fraud?
- **Recall**: of all actual frauds, how many did we catch?
- **F1 Score**: the harmonic mean of precision and recall
- **ROC-AUC**: overall ability to distinguish fraud from legit

In [ ]:
# ============================================================
# CELL 22 — PREDICTIONS FOR ALL THREE MODELS
# ============================================================
# Generate predictions and probability scores for each model.
# predict() → hard labels (0 or 1)
# predict_proba()[:,1] → probability of being fraud (used for ROC/AUC)

y_pred_rf  = rf_model.predict(X_test)
y_pred_lr  = lr_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)

y_prob_rf  = rf_model.predict_proba(X_test)[:, 1]
y_prob_lr  = lr_model.predict_proba(X_test)[:, 1]
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions generated for all 3 models")

In [ ]:
# ============================================================
# CELL 23 — MODEL COMPARISON TABLE 
# ============================================================
# This summary table is the most interview-friendly thing in the whole notebook.
# It lets you immediately compare all three models on the metrics that matter.
# F1 and AUC are far more meaningful than accuracy for fraud detection.

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)

results = {
    'Model': ['Random Forest', 'Logistic Regression', 'XGBoost'],
    'Accuracy':  [accuracy_score(y_test, y_pred_rf),
                  accuracy_score(y_test, y_pred_lr),
                  accuracy_score(y_test, y_pred_xgb)],
    'Precision': [precision_score(y_test, y_pred_rf),
                  precision_score(y_test, y_pred_lr),
                  precision_score(y_test, y_pred_xgb)],
    'Recall':    [recall_score(y_test, y_pred_rf),
                  recall_score(y_test, y_pred_lr),
                  recall_score(y_test, y_pred_xgb)],
    'F1 Score':  [f1_score(y_test, y_pred_rf),
                  f1_score(y_test, y_pred_lr),
                  f1_score(y_test, y_pred_xgb)],
    'ROC-AUC':   [roc_auc_score(y_test, y_prob_rf),
                  roc_auc_score(y_test, y_prob_lr),
                  roc_auc_score(y_test, y_prob_xgb)],
}

results_df = pd.DataFrame(results).set_index('Model').round(4)
print("=== Model Comparison ===")
results_df

In [ ]:
# ============================================================
# CELL 24 — CONFUSION MATRICES FOR ALL THREE MODELS 
# ============================================================
# A confusion matrix shows True Positives, False Positives, etc.
# In fraud detection, False Negatives (missed fraud) are the most costly.
# Showing all three side by side makes for a clean, professional comparison.

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models_info = [
    ('Random Forest',       y_pred_rf),
    ('Logistic Regression', y_pred_lr),
    ('XGBoost',             y_pred_xgb),
]

for ax, (name, preds) in zip(axes, models_info):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 25 — ROC CURVES 
# ============================================================
# The ROC Curve plots True Positive Rate vs False Positive Rate at every
# classification threshold. AUC (Area Under Curve) summarizes this in one number.
# A perfect model has AUC = 1.0. A random guess gives AUC = 0.5.
# This is the gold standard chart for binary classifiers in interviews.

from sklearn.metrics import roc_curve

plt.figure(figsize=(9, 6))

for name, probs in [('Random Forest', y_prob_rf),
                     ('Logistic Regression', y_prob_lr),
                     ('XGBoost', y_prob_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Guess')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Observation:** The ROC curve shows how well each model separates fraud from legitimate transactions across all thresholds. A higher AUC means the model is better at ranking fraudulent transactions above legitimate ones.

In [ ]:
# ============================================================
# CELL 26 — PRECISION-RECALL CURVES 
# ============================================================
# For highly imbalanced datasets, Precision-Recall curves are even more
# informative than ROC curves. They focus specifically on how well the model
# performs on the minority (fraud) class — which is what we actually care about.
# A high Average Precision score means the model catches fraud with few false alarms.

from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(9, 6))

for name, probs in [('Random Forest', y_prob_rf),
                     ('Logistic Regression', y_prob_lr),
                     ('XGBoost', y_prob_xgb)]:
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    plt.plot(rec, prec, linewidth=2, label=f'{name} (AP = {ap:.3f})')

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves — All Models', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 27 — DETAILED CLASSIFICATION REPORTS
# ============================================================
# Full breakdown of Precision, Recall, F1 per class for each model.
# Helps explain in an interview *which model is best and why*.

from sklearn.metrics import classification_report

for name, preds in [('Random Forest', y_pred_rf),
                     ('Logistic Regression', y_pred_lr),
                     ('XGBoost', y_pred_xgb)]:
    print(f"{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(classification_report(y_test, preds, target_names=['Legitimate', 'Fraud']))
    print()

---
## 7. Feature Importance

In [ ]:
# ============================================================
# CELL 28 — FEATURE IMPORTANCE (RANDOM FOREST) 
# ============================================================
# Random Forest automatically calculates how useful each feature was
# for making predictions. Higher importance = more useful.
# This chart is a great talking point in interviews — you can explain
# *why* certain features (like amount, distance) matter more than others.

feat_importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top_features = feat_importances.nlargest(15)

plt.figure(figsize=(10, 6))
bars = plt.barh(top_features.index[::-1], top_features.values[::-1],
                color=sns.color_palette('Blues_d', 15), edgecolor='black')
plt.xlabel('Feature Importance Score')
plt.title('Top 15 Most Important Features — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 15 Features:")
print(top_features.round(4).to_string())

**Observation:** The feature importance chart confirms which signals are most predictive of fraud. Features like `amount`, `distance`, `trans_hour`, and `age` tend to rank highly — consistent with what we saw in the EDA.

---
## 8. Save the Model

In [ ]:
# ============================================================
# CELL 29 — SAVE THE BEST MODEL 
# ============================================================
# We save the trained model to disk using joblib.
# This is what a production pipeline would do — train once, save, then
# load the saved model in an API or app without retraining every time.
# We also save the list of feature columns so the Streamlit app knows
# exactly what input format the model expects.

import joblib

joblib.dump(rf_model, 'fraud_model.pkl')
joblib.dump(X_train.columns.tolist(), 'model_columns.pkl')

print("Model saved as fraud_model.pkl")
print("Column names saved as model_columns.pkl")
print(f"   Total features: {len(X_train.columns)}")

---
## 9. Conclusion

In this project, we built an end-to-end credit card fraud detection system using real-world transaction data.

**Key steps completed:**
- Cleaned and explored the dataset, identifying class imbalance and key fraud patterns
- Engineered meaningful features: transaction hour, cardholder age, merchant distance, is_night flag, and log-transformed city population
- Correctly handled class imbalance by upsampling *only on the training set* to avoid data leakage
- Trained and compared three models: Logistic Regression (baseline), Random Forest, and XGBoost
- Evaluated models using F1 Score, ROC-AUC, and Precision-Recall curves — metrics appropriate for imbalanced fraud data

**Best model:** Random Forest / XGBoost (see comparison table above for exact scores)

**Key insight:** Transaction amount, distance between cardholder and merchant, and the hour of the transaction were among the strongest predictors of fraud.

> 🚀 **Deployment:** This model is deployed as an interactive web application using **Streamlit** — see `app.py` in this repository. Users can enter transaction details and receive a real-time fraud prediction with a confidence score.